# Week 26, Day 4: Review IV: probability, statistics and Bayesian inference

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w026/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: a firing rate and its uncertainty

A neuron gives Poisson spike counts on 40 trials. Compute the maximum likelihood rate `lam_hat`, its standard error from the Fisher information, `se_fisher` = √(λ̂/n), and a bootstrap standard error `se_boot` from 2,000 resamples of the trials (local rng seed 13).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
counts = np.random.default_rng(15).poisson(6.0, 40)
lam_hat = ...
se_fisher = ...
rb = np.random.default_rng(13)
boot = [rb.choice(counts, size=len(counts), replace=True).mean() for _ in range(2000)]
se_boot = ...
print(f"rate {lam_hat:.2f} spikes per trial;  SE from Fisher information {se_fisher:.3f}, from the bootstrap {se_boot:.3f};  Fano factor {counts.var(ddof=1) / counts.mean():.2f}")

In [ ]:
import numpy as np
def test_mle():
    "lam_hat is the mean count and se_fisher is sqrt(lam_hat / n)"
    assert abs(lam_hat - counts.mean()) < 1e-12 and abs(se_fisher - np.sqrt(lam_hat / len(counts))) < 1e-12

def test_boot():
    "the bootstrap standard error agrees with the Fisher one within 15%"
    assert abs(se_boot - se_fisher) / se_fisher < 0.15

test_mle()
test_boot()
print("All checks passed")

### Exercise: Problem 2: fifty neurons, honest discoveries

Fifty neurons are recorded on 30 stimulus and 30 blank trials; the first 10 truly respond. For each neuron, compute a permutation p-value for the difference in mean count (1,000 shuffles, local rng seed 18, one-sided, (1 + count) / 1001). Compare the uncorrected decisions `naive` (p below 0.05) with Benjamini-Hochberg at q = 0.05, `bh_reject`.

In [ ]:
rd = np.random.default_rng(16)
truly = np.arange(50) < 10
stim = rd.poisson(np.where(truly, 6.5, 4.0)[:, None], (50, 30))
blank = rd.poisson(4.0, (50, 30))
rp = np.random.default_rng(18)
pvals = np.empty(50)
for n in range(50):
    both = np.concatenate([stim[n], blank[n]]); obs = stim[n].mean() - blank[n].mean()
    null = np.empty(1000)
    for k in range(1000):
        sh = rp.permutation(both); null[k] = sh[:30].mean() - sh[30:].mean()
    pvals[n] = ...
naive = pvals < 0.05
order = np.argsort(pvals); m = 50
passed = np.where(np.sort(pvals) <= 0.05 * np.arange(1, m + 1) / m)[0]
bh_reject = np.zeros(m, bool)
if len(passed):
    bh_reject[...] = True
print(f"uncorrected: {naive.sum()} discoveries, {(naive & ~truly).sum()} false;  BH: {bh_reject.sum()} discoveries, {(bh_reject & ~truly).sum()} false")

In [ ]:
import numpy as np
def test_p():
    "pvals holds one permutation p-value per neuron, between 1/1001 and 1"
    assert len(pvals) == 50 and np.all(pvals >= 1 / 1001) and np.all(pvals <= 1)

def test_bh():
    "bh_reject follows the Benjamini-Hochberg step-up rule at q = 0.05"
    order = np.argsort(pvals); m = len(pvals)
    ok = np.where(np.sort(pvals) <= 0.05 * np.arange(1, m + 1) / m)[0]
    ref = np.zeros(m, bool)
    if len(ok): ref[order[:ok.max() + 1]] = True
    assert np.array_equal(bh_reject, ref)

def test_fewer():
    "BH makes at most as many discoveries as the uncorrected threshold, with no more false ones"
    assert bh_reject.sum() <= naive.sum() and (bh_reject & ~truly).sum() <= (naive & ~truly).sum()

test_p()
test_bh()
test_fewer()
print("All checks passed")

### Exercise: Problem 3: a posterior exactly and by sampling

With a Gamma(a0 = 2, b0 = 0.5) prior on the rate and Problem 1's counts, the posterior is Gamma(a0 + Σk, b0 + n). Store its mean in `post_mean`. Then sample the same posterior with Metropolis (random-walk proposals of sd 0.3, 20,000 steps from λ = 1, local rng seed 19, discarding the first 2,000), using only the unnormalised log posterior, and store the samples in `samples`.

In [ ]:
a0, b0 = 2.0, 0.5
post_mean = ...

def log_post(lam):
    if lam <= 0: return -np.inf
    return (a0 - 1) * np.log(lam) - b0 * lam + counts.sum() * np.log(lam) - len(counts) * lam

rm = np.random.default_rng(19)
lam, lp, chain = 1.0, log_post(1.0), []
for _ in range(20000):
    prop = lam + 0.3 * rm.normal()
    lp_prop = log_post(prop)
    if ...:                                      # accept with probability min(1, exp(lp_prop - lp))
        lam, lp = prop, lp_prop
    chain.append(lam)
samples = np.array(chain[2000:])
print(f"exact posterior mean {post_mean:.3f};  Metropolis {samples.mean():.3f} (sd {samples.std():.3f});  95% interval {np.percentile(samples, [2.5, 97.5]).round(2)}")

In [ ]:
import numpy as np
def test_conj():
    "the conjugate posterior is Gamma(a0 + sum counts, b0 + n) with this mean"
    assert abs(post_mean - (a0 + counts.sum()) / (b0 + len(counts))) < 1e-12

def test_mcmc():
    "the Metropolis samples' mean and sd agree with the exact posterior within 3%"
    a, b = a0 + counts.sum(), b0 + len(counts)
    assert abs(samples.mean() - a / b) / (a / b) < 0.03 and abs(samples.std() - np.sqrt(a) / b) / (np.sqrt(a) / b) < 0.03

test_conj()
test_mcmc()
print("All checks passed")